In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------
# 1. Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
import gng_py

In [ ]:
# -------------------------------------------------
# 2. Load & preprocess data
# -------------------------------------------------
iris = load_iris()
X = iris.data
y = iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_gpu = scaler.fit_transform(X_train)
X_test_gpu = scaler.transform(X_test)

# Convert to tensors
X_train_gpu = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_train_gpu = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)


In [ ]:
print(X_train.flatten())

In [ ]:
import time
config_file = "config_iris.json"
ctx = gng_py.PyContext()

ctx = gng_py.PyContext()
ctx.create_system()
ctx.load_config(config_file)
ctx.init_dataset_vec(X_train.flatten())
start = time.time()
ctx.fit()
end = time.time()
model_string = ctx.get_model_string()
print("Time for gng calculation: ",end-start)

In [ ]:
import json
import pandas as pd
data = json.loads(model_string)

In [ ]:
points = None
edges = None
edge_positions = None
rows = []
for neuron in data["model"]["neurons"]:
    position = neuron["position"]
    id = neuron['id']
    x = position[0]
    y = position[1]
    rows.append({"id": id, "x": x, "y": y})

points = pd.DataFrame(rows)

In [ ]:
data

In [ ]:
for a in data["model"]["neurons"]:
    a["class"] = 0
    print(a)

In [ ]:
df = pd.DataFrame(data["model"]["neurons"])
df

In [ ]:
df.iloc[98].position

In [ ]:


# -------------------------------------------------
# 3. Define a simple MLP model
# -------------------------------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -------------------------------------------------
# 4. Loss and optimizer
# -------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -------------------------------------------------
# 5. Training loop
# -------------------------------------------------
epochs = 300
batch_size = 8

dataset = torch.utils.data.TensorDataset(X_train, y_train)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(loader):.4f}")

# -------------------------------------------------
# 6. Evaluation
# -------------------------------------------------
model.eval()
with torch.no_grad():
    preds = model(X_test)
    correct = (preds.argmax(dim=1) == y_test).sum().item()
    acc = correct / len(y_test)

print(f"Test accuracy: {acc:.4f}")
